# 11 — Tool Calling and Tool Interface Design

## Scenario
Northstar can read order status or draft a refund request, but it cannot execute a refund directly. 
We need to give the model the ability to trigger a function in our application code to look up an order.

**The Concept:** Tool Calling (or Function Calling) is how LLMs interact with the outside world. The model doesn't run the code; it outputs a structured JSON request (a `function_call`) asking *your application* to run the code.

In [ ]:
import os
from google import genai
from google.genai import types

# Initialize the client (requires GEMINI_API_KEY environment variable)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'


## Step 1: Defining the Tool Interface

We define a standard Python function. The `google-genai` SDK automatically inspects the type hints and the docstring to generate the underlying JSON schema that the model understands.

In [ ]:
def get_order_status(order_id: str) -> str:
    """Retrieves the current shipping status of a Northstar order.\n    \n    Args:\n        order_id: The unique identifier for the order (e.g., 'ORD-123').\n    """
    # In a real application, this would query a database.
    print(f"\n[SYSTEM] Executing get_order_status for {order_id}...")
    mock_db = {
        "ORD-123": "Shipped - Arriving Tuesday",
        "ORD-999": "Processing - Delayed by 2 days"
    }
    return mock_db.get(order_id, "Order not found")


## Step 2: Triggering the Tool Call

We pass the tool to the model and ask a question. Notice how we do **not** enable automatic execution. We want to inspect the manual loop to understand the security boundary.

In [ ]:
user_message = "Where is my order ORD-999?"

# 1. Send the initial request with the tool
response_1 = client.models.generate_content(
    model=MODEL_ID,
    contents=user_message,
    config=types.GenerateContentConfig(
        tools=[get_order_status],
        temperature=0.0
    )
)

print("--- Model's Tool Call Request ---")
# The model didn't return text; it returned a function_call object!
function_call = response_1.function_calls[0]
print(f"Function requested: {function_call.name}")
print(f"Arguments provided: {function_call.args}")


## Step 3: Executing and Returning the Result

The application (us) now executes the actual Python code and sends the result back to the model so it can answer the user.

In [ ]:
# 2. The Application executes the function
args = function_call.args
result = get_order_status(order_id=args["order_id"])
print(f"[SYSTEM] Function returned: {result}\n")

# 3. We send the result back to the model
conversation_history = [
    types.Content(role="user", parts=[types.Part.from_text(user_message)]),
    response_1.candidates[0].content, # The model's function_call
    types.Content(
        role="user",
        parts=[
            types.Part.from_function_response(
                name=function_call.name,
                response={"result": result}
            )
        ]
    )
]

final_response = client.models.generate_content(
    model=MODEL_ID,
    contents=conversation_history,
    config=types.GenerateContentConfig(tools=[get_order_status])
)

print("--- Final Model Output ---")
print(final_response.text)


## Conclusion

By controlling the manual execution loop, the application maintains ultimate authority over security and authorization. If the tool was `execute_refund()`, the application could pause here, ask a human for approval, and only then return the `function_response` to the model.